# F1 Strategy: GRU Multi-Task Model + Strategy Simulator (OpenF1 + FastF1)

This notebook combines:
- GRU + MLP multi-task model (stops cls, tire cls, time reg)
- Strategy enumeration/simulation (0..3 stops)
- Ranked output with total predicted race time and stint breakdown
- Train/validation plots for loss and accuracy

Upgrade vs original:
- Training dataset is **OpenF1 + FastF1** unified into the same schema.
- FastF1 data is loaded via `fastf1` and converted into the same columns and weather sequence format.


In [1]:
import re
import math
from pathlib import Path
from itertools import product
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

import fastf1
from fastf1.core import Session
from fastf1 import plotting

pd.set_option('display.max_columns', 200)
np.random.seed(42)


In [2]:
# --------------------
# Configuration
# --------------------
DATA_DIR = Path("openf1_data")
assert DATA_DIR.exists(), f"Missing: {DATA_DIR.resolve()}"

# FastF1 cache (recommended)
FASTF1_CACHE_DIR = Path("fastf1_cache")
FASTF1_CACHE_DIR.mkdir(exist_ok=True)
fastf1.Cache.enable_cache(str(FASTF1_CACHE_DIR))

MODEL_FILE = "multitask_strategy_model_openf1_fastf1.keras"
PREPROC_FILE = "multitask_preprocessing_openf1_fastf1.joblib"

# Inference / simulator inputs
RACE_CONTEXT_FILE = "race_context_input.csv"
FORECAST_FILE = "forecast_weather_detailed_rainy.csv"

WEATHER_FEATURES = ["air_temperature", "track_temperature", "humidity", "rainfall", "wind_speed"]

VALID_COMPOUNDS = ["SOFT", "MEDIUM", "HARD", "INTERMEDIATE", "WET"]
DRY_COMPOUNDS = ["SOFT", "MEDIUM", "HARD"]

DEFAULT_PIT_LOSS = 22.0  # seconds

# FastF1 collection range
# Use all available data in this range; increase if you want more.
FASTF1_START_YEAR = 2018
FASTF1_END_YEAR = datetime.utcnow().year

# Which sessions to use from FastF1; we primarily want races.
FASTF1_SESSION_NAME = "R"  # Race

# To limit runtime in early experiments you can set e.g. 5; keep None for all
FASTF1_MAX_SESSIONS = None


In [3]:
def parse_session_key(path: Path):
    m = re.search(r"_session_(\d+)\.csv$", path.name)
    return int(m.group(1)) if m else None

def safe_read_csv(path: Path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.DataFrame()

def safe_mode(series, default="MEDIUM"):
    s = pd.Series(series).dropna()
    if s.empty:
        return default
    m = s.mode()
    return m.iloc[0] if not m.empty else default

def format_time(sec):
    m = int(sec // 60)
    s = sec - m * 60
    return f"{m}:{s:06.3f}"

def ensure_weather_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in WEATHER_FEATURES:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def compute_weather_seq_and_summary(weather_df: pd.DataFrame):
    if weather_df is None or len(weather_df) == 0:
        return None, None
    w = ensure_weather_columns(weather_df)
    w = w.dropna(subset=WEATHER_FEATURES, how="all")
    if w.empty:
        return None, None

    seq = w[WEATHER_FEATURES].ffill().bfill().fillna(0.0).values
    summary = {
        "air_temp_mean": float(w["air_temperature"].mean()),
        "track_temp_mean": float(w["track_temperature"].mean()),
        "humidity_mean": float(w["humidity"].mean()),
        "rain_minutes_ratio": float((w["rainfall"].fillna(0) > 0).mean()),
        "wind_speed_mean": float(w["wind_speed"].mean()),
    }
    return seq, summary

def normalize_compound(name: str, default="MEDIUM"):
    if name is None or (isinstance(name, float) and np.isnan(name)):
        return default
    s = str(name).upper().strip()
    # FastF1 usually uses e.g. 'SOFT', 'MEDIUM', 'HARD', 'INTERMEDIATE', 'WET'
    if s not in VALID_COMPOUNDS:
        return default
    return s

def estimate_pit_loss_from_fastf1_pits(pit_laps_df: pd.DataFrame):
    """Estimate total pit time loss from FastF1.

    FastF1 does not always provide a single 'pit time loss'. We approximate:
    - If PitInTime/PitOutTime are available per stop, sum (PitOut - PitIn)
    - Otherwise fallback to DEFAULT_PIT_LOSS per stop
    Then add a fixed additional loss per stop (e.g., pitlane delta) if needed.
    Here we keep it simple and treat (PitOut-PitIn) as the full loss; if missing, use DEFAULT_PIT_LOSS.
    """
    if pit_laps_df is None or len(pit_laps_df) == 0:
        return 0.0

    # Keep only rows that are pit stops
    pits = pit_laps_df[pit_laps_df.get("PitInTime").notna()]
    if len(pits) == 0:
        return 0.0

    if "PitInTime" in pits.columns and "PitOutTime" in pits.columns:
        # Both are timedelta-like; convert to seconds
        try:
            delta = (pits["PitOutTime"] - pits["PitInTime"]).dt.total_seconds()
            delta = pd.to_numeric(delta, errors="coerce").dropna()
            if len(delta) > 0:
                # Some deltas might be 0/negative or unrealistic; clamp
                delta = delta.clip(lower=0)
                return float(delta.sum())
        except Exception:
            pass

    # Fallback
    return float(DEFAULT_PIT_LOSS * len(pits))


In [4]:
def load_session_maps_openf1():
    sessions = safe_read_csv(DATA_DIR / "sessions.csv")
    if sessions.empty:
        return {}, {}, {}

    for c in ["session_key", "meeting_key", "session_name", "location", "circuit_short_name"]:
        if c not in sessions.columns:
            sessions[c] = np.nan

    sessions["track_name"] = (
        sessions["circuit_short_name"].fillna(sessions["location"]).fillna(sessions["meeting_key"].astype(str))
    )

    return (
        dict(zip(sessions["session_key"], sessions["track_name"])),
        dict(zip(sessions["session_key"], sessions["meeting_key"])),
        dict(zip(sessions["session_key"], sessions["session_name"].astype(str))),
    )

session_to_track_openf1, session_to_meeting_openf1, session_to_name_openf1 = load_session_maps_openf1()

## Build training data (OpenF1 + FastF1 unified)

We will build **two** datasets:
- OpenF1: using local CSVs (same as original)
- FastF1: using `fastf1` API

Then we concatenate and train the same model.


In [5]:
def get_weather_seq_and_summary_openf1(session_key):
    w = safe_read_csv(DATA_DIR / f"weather_session_{session_key}.csv")
    if w.empty:
        return None, None
    w = ensure_weather_columns(w)
    return compute_weather_seq_and_summary(w)

def estimate_time_loss_target_openf1(session_key, team_name, driver_team):
    p = safe_read_csv(DATA_DIR / f"pit_session_{session_key}.csv")
    if p.empty or "driver_number" not in p.columns:
        return 0.0
    p = p.merge(driver_team, on="driver_number", how="left")
    p = p[p["team_name"] == team_name]
    if p.empty:
        return 0.0
    for c in ["pit_duration", "duration", "pit_time"]:
        if c in p.columns:
            d = pd.to_numeric(p[c], errors="coerce").dropna()
            if len(d):
                return float(d.sum() + 15.0 * len(d))
    return float(21.0 * len(p))

def build_samples_openf1(race_only=True):
    rows, seqs = [], []
    for sf in sorted(DATA_DIR.glob("stints_session_*.csv")):
        sid = parse_session_key(sf)
        if sid is None:
            continue
        sname = session_to_name_openf1.get(sid, "")
        if race_only and ("Race" not in sname and "RACE" not in sname):
            continue

        st = safe_read_csv(sf)
        dr = safe_read_csv(DATA_DIR / f"drivers_session_{sid}.csv")
        if st.empty or dr.empty:
            continue

        for c in ["driver_number", "compound", "stint_number"]:
            if c not in st.columns:
                st[c] = np.nan
        for c in ["driver_number", "team_name"]:
            if c not in dr.columns:
                dr[c] = np.nan

        driver_team = dr[["driver_number", "team_name"]].dropna().drop_duplicates()
        st = st.merge(driver_team, on="driver_number", how="left")

        pits = safe_read_csv(DATA_DIR / f"pit_session_{sid}.csv")
        if pits.empty:
            pits = pd.DataFrame(columns=["driver_number"])
        if "driver_number" not in pits.columns:
            pits["driver_number"] = np.nan
        pits = pits.merge(driver_team, on="driver_number", how="left")

        grid = safe_read_csv(DATA_DIR / f"starting_grid_session_{sid}.csv")
        pos_col = "position" if "position" in grid.columns else ("grid_position" if "grid_position" in grid.columns else None)
        if not grid.empty and pos_col and "driver_number" in grid.columns:
            g = grid[["driver_number", pos_col]].copy()
            g.columns = ["driver_number", "starting_position"]
            g["starting_position"] = pd.to_numeric(g["starting_position"], errors="coerce")
            g = g.merge(driver_team, on="driver_number", how="left")
            team_pos = g.groupby("team_name")["starting_position"].mean().to_dict()
        else:
            team_pos = {}

        seq, ws = get_weather_seq_and_summary_openf1(sid)
        if seq is None:
            continue

        track = str(session_to_track_openf1.get(sid, session_to_meeting_openf1.get(sid, "UNKNOWN_TRACK")))

        for team, t_st in st.groupby("team_name", dropna=True):
            team = str(team)
            t_pits = pits[pits["team_name"] == team]
            pit_stops = int(len(t_pits))

            s1 = t_st[t_st["stint_number"] == 1]
            start_comp = safe_mode(s1["compound"] if len(s1) else t_st["compound"], default="MEDIUM")
            start_comp = normalize_compound(start_comp, default="MEDIUM")

            non_start = t_st[t_st["compound"].astype(str).str.upper() != start_comp]["compound"]
            target_comp = normalize_compound(
                safe_mode(non_start, default=safe_mode(t_st["compound"], "MEDIUM")),
                default="MEDIUM",
            )

            rows.append({
                "source": "openf1",
                "session_key": sid,
                "team_name": team,
                "track_name": track,
                "starting_position": float(team_pos.get(team, 10.0)),
                "starting_compound": start_comp,
                "pit_stops": pit_stops,
                "target_compound": target_comp,
                "time_loss_target": estimate_time_loss_target_openf1(sid, team, driver_team),
                **ws,
            })
            seqs.append(seq)

    return pd.DataFrame(rows), seqs


In [6]:
def fastf1_track_name(ev: pd.Series):
    # Prefer EventName; fallback to Location
    for c in ["EventName", "Location", "OfficialEventName"]:
        if c in ev.index and pd.notna(ev[c]):
            return str(ev[c])
    return "UNKNOWN_TRACK"

def build_samples_fastf1(start_year=FASTF1_START_YEAR, end_year=FASTF1_END_YEAR, session_name=FASTF1_SESSION_NAME, max_sessions=FASTF1_MAX_SESSIONS):
    """Build samples from FastF1 in the *same schema* as OpenF1 samples.

    Sample granularity is team-per-race (like OpenF1 team grouping).

    Columns returned match OpenF1's `df` columns.
    Weather sequences are taken from FastF1 weather data.
    """
    rows, seqs = [], []

    schedule_years = list(range(int(start_year), int(end_year) + 1))
    collected_sessions = 0

    for year in schedule_years:
        try:
            schedule = fastf1.get_event_schedule(year)
        except Exception as e:
            print(f"[FastF1] Could not load schedule for {year}: {e}")
            continue

        if schedule is None or len(schedule) == 0:
            continue

        for _, ev in schedule.iterrows():
            if max_sessions is not None and collected_sessions >= max_sessions:
                break

            # Only include events that have a race session
            try:
                ses = fastf1.get_session(year, ev["EventName"], session_name)
            except Exception:
                continue

            # CRITICAL: load required data BEFORE accessing properties
            try:
                # weather=True ensures session.weather_data is available
                # laps=True ensures session.laps is available
                ses.load(laps=True, weather=True)
            except Exception as e:
                print(f"[FastF1] load failed {year} {ev.get('EventName', '')}: {e}")
                continue

            # Laps
            try:
                laps = ses.laps
            except Exception:
                continue

            if laps is None or len(laps) == 0:
                continue

            # Weather
            weather = getattr(ses, "weather_data", None)
            if weather is None or len(weather) == 0:
                # Some sessions have no weather; skip to keep schema consistent with GRU weather seq
                continue

            # FastF1 columns are typically: AirTemp, TrackTemp, Humidity, Rainfall, WindSpeed
            w = weather.copy()
            rename_map = {
                "AirTemp": "air_temperature",
                "TrackTemp": "track_temperature",
                "Humidity": "humidity",
                "Rainfall": "rainfall",
                "WindSpeed": "wind_speed",
            }
            for src, dst in rename_map.items():
                if src in w.columns and dst not in w.columns:
                    w[dst] = w[src]

            seq, ws = compute_weather_seq_and_summary(w)
            if seq is None:
                continue

            track = fastf1_track_name(ev)

            # Join driver/team information
            # Laps has 'Team' and 'DriverNumber' or 'Driver'
            if "Team" not in laps.columns:
                continue

            # Determine starting compounds per driver from first stint laps
            # FastF1 laps includes 'Compound' when tyres are available
            if "Compound" not in laps.columns:
                continue

            # Starting grid positions: use session.results if available
            team_start_pos = {}
            try:
                res = getattr(ses, "results", None)
                if res is not None and len(res) > 0 and "TeamName" in res.columns:
                    pos_col = "GridPosition" if "GridPosition" in res.columns else ("GridPosition" if "GridPosition" in res.columns else None)
                    if pos_col is None and "GridPosition" in res.columns:
                        pos_col = "GridPosition"
                    if pos_col is None and "GridPosition" not in res.columns and "GridPosition" in res.columns:
                        pos_col = "GridPosition"
                    # FastF1 results uses 'GridPosition'
                    pos_col = "GridPosition" if "GridPosition" in res.columns else None
                    if pos_col:
                        tmp = res[["TeamName", pos_col]].copy()
                        tmp[pos_col] = pd.to_numeric(tmp[pos_col], errors="coerce")
                        team_start_pos = tmp.groupby("TeamName")[pos_col].mean().to_dict()
            except Exception:
                team_start_pos = {}

            # For each team, compute pit stops, start compound, target compound, and time loss target
            for team, t_laps in laps.groupby("Team", dropna=True):
                team = str(team)
                t_laps = t_laps.sort_values("LapNumber")

                # Pit stops count: count distinct PitInTime occurrences
                pit_laps = t_laps[t_laps.get("PitInTime").notna()] if "PitInTime" in t_laps.columns else pd.DataFrame()
                pit_stops = int(len(pit_laps))

                # Start compound: compound on first lap of each driver, then mode across drivers
                start_compounds = []
                target_compounds = []

                # Group by driver if possible
                drv_col = "DriverNumber" if "DriverNumber" in t_laps.columns else ("Driver" if "Driver" in t_laps.columns else None)
                if drv_col:
                    for _, d_laps in t_laps.groupby(drv_col, dropna=True):
                        d_laps = d_laps.sort_values("LapNumber")
                        first = d_laps.head(1)
                        if len(first) and "Compound" in first.columns:
                            start_compounds.append(normalize_compound(first["Compound"].iloc[0], default="MEDIUM"))

                        # find any compounds different from starting for that driver
                        if "Compound" in d_laps.columns and len(d_laps) > 0:
                            comps = d_laps["Compound"].dropna().astype(str).str.upper().tolist()
                            if len(comps) > 0:
                                sc = normalize_compound(comps[0], default="MEDIUM")
                                others = [normalize_compound(c, default=sc) for c in comps if normalize_compound(c, default=sc) != sc]
                                if len(others) > 0:
                                    target_compounds.extend(others)
                else:
                    # fallback: use all laps
                    sc = safe_mode(t_laps["Compound"].dropna(), default="MEDIUM")
                    sc = normalize_compound(sc)
                    start_compounds = [sc]
                    others = t_laps["Compound"].dropna().astype(str).str.upper()
                    others = [normalize_compound(c, default=sc) for c in others if normalize_compound(c, default=sc) != sc]
                    target_compounds = others

                start_comp = normalize_compound(safe_mode(start_compounds, default="MEDIUM"), default="MEDIUM")
                target_comp = normalize_compound(safe_mode(target_compounds, default=start_comp), default="MEDIUM")

                # time loss target estimation
                time_loss_target = estimate_pit_loss_from_fastf1_pits(pit_laps)

                rows.append({
                    "source": "fastf1",
                    "session_key": f"fastf1_{year}_{ev.get('RoundNumber', '')}_{ev.get('EventName', '')}",
                    "team_name": team,
                    "track_name": track,
                    "starting_position": float(team_start_pos.get(team, 10.0)),
                    "starting_compound": start_comp,
                    "pit_stops": pit_stops,
                    "target_compound": target_comp,
                    "time_loss_target": float(time_loss_target),
                    **ws,
                })
                seqs.append(seq)

            collected_sessions += 1

    return pd.DataFrame(rows), seqs


In [7]:
print("Building OpenF1 samples...")
df_openf1, X_seq_openf1 = build_samples_openf1(race_only=True)
print("OpenF1 samples:", len(df_openf1))

print("\nBuilding FastF1 samples...")
df_fastf1, X_seq_fastf1 = build_samples_fastf1()
print("FastF1 samples:", len(df_fastf1))

# Combine
df = pd.concat([df_openf1, df_fastf1], ignore_index=True)
X_seq_raw = list(X_seq_openf1) + list(X_seq_fastf1)

print("\nCombined samples:", len(df))
print(df.head())
print("\nSources:")
print(df['source'].value_counts(dropna=False))


Building OpenF1 samples...


core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


OpenF1 samples: 90

Building FastF1 samples...


core        WARNING 	Driver 5 completed the race distance 00:00.123000 before the recorded end of the session.
req            INFO 	No cached data found for car_data. Loading data...
_api           INFO 	Fetching car data...
_api           INFO 	Parsing car data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for position_data. Loading data...
_api           INFO 	Fetching position data...
core        WARNING 	Car position data is unavailable!
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['5', '44', '7', '3', '14', '33', '27', '77', '2', '55', '11', '31', '16', '18', '28', '8', '20', '10', '9', '35']
core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Us

[FastF1] load failed 2021 Mexico City Grand Prix: any API: 500 calls/h


...

[FastF1] load failed 2021 Qatar Grand Prix: any API: 500 calls/h


Request for URL https://raw.githubusercontent.com/theOehrly/f1schedule/master/schedule_2021.json failed; using cached response
Traceback (most recent call last):
  File "/Users/m/miniforge3/envs/tensorflow/lib/python3.10/site-packages/requests_cache/session.py", line 311, in _resend
    response = self._send_and_cache(request, actions, cached_response, **kwargs)
  File "/Users/m/miniforge3/envs/tensorflow/lib/python3.10/site-packages/requests_cache/session.py", line 279, in _send_and_cache
    response = super().send(request, **kwargs)
  File "/Users/m/miniforge3/envs/tensorflow/lib/python3.10/site-packages/fastf1/req.py", line 110, in send
    lim.limit()
  File "/Users/m/miniforge3/envs/tensorflow/lib/python3.10/site-packages/fastf1/req.py", line 80, in limit
    raise RateLimitExceededError(self._info)
fastf1.exceptions.RateLimitExceededError: any API: 500 calls/h
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found

[FastF1] load failed 2021 Saudi Arabian Grand Prix: any API: 500 calls/h


Request for URL https://raw.githubusercontent.com/theOehrly/f1schedule/master/schedule_2021.json failed; using cached response
Traceback (most recent call last):
  File "/Users/m/miniforge3/envs/tensorflow/lib/python3.10/site-packages/requests_cache/session.py", line 311, in _resend
    response = self._send_and_cache(request, actions, cached_response, **kwargs)
  File "/Users/m/miniforge3/envs/tensorflow/lib/python3.10/site-packages/requests_cache/session.py", line 279, in _send_and_cache
    response = super().send(request, **kwargs)
  File "/Users/m/miniforge3/envs/tensorflow/lib/python3.10/site-packages/fastf1/req.py", line 110, in send
    lim.limit()
  File "/Users/m/miniforge3/envs/tensorflow/lib/python3.10/site-packages/fastf1/req.py", line 80, in limit
    raise RateLimitExceededError(self._info)
fastf1.exceptions.RateLimitExceededError: any API: 500 calls/h
core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for

[FastF1] load failed 2021 Abu Dhabi Grand Prix: any API: 500 calls/h
[FastF1] Could not load schedule for 2022: any API: 500 calls/h
[FastF1] Could not load schedule for 2023: any API: 500 calls/h
[FastF1] Could not load schedule for 2024: any API: 500 calls/h
[FastF1] Could not load schedule for 2025: any API: 500 calls/h
[FastF1] Could not load schedule for 2026: any API: 500 calls/h
FastF1 samples: 780

Combined samples: 870
   source session_key     team_name track_name  starting_position  \
0  openf1       10033        Alpine      Miami               10.0   
1  openf1       10033  Aston Martin      Miami               10.0   
2  openf1       10033       Ferrari      Miami               10.0   
3  openf1       10033  Haas F1 Team      Miami               10.0   
4  openf1       10033   Kick Sauber      Miami               10.0   

  starting_compound  pit_stops target_compound  time_loss_target  \
0              HARD          1          MEDIUM            37.060   
1              HA

## Preprocessing (same for combined dataset)

We keep the same preprocessing approach as your improved notebook:
- Label encoding
- Robust numeric standardization
- Weather sequence padding + normalization
- Log-normalization for time target


In [8]:
print("\nPreprocessing...")

# Label encoders
team_le = LabelEncoder()
track_le = LabelEncoder()
start_le = LabelEncoder()
tire_le = LabelEncoder()
stops_le = LabelEncoder()

team_le.fit(df["team_name"].astype(str))
track_le.fit(df["track_name"].astype(str))
start_le.fit(df["starting_compound"].astype(str))
tire_le.fit(df["target_compound"].astype(str))
stops_le.fit(df["pit_stops"].astype(str))

X_team = team_le.transform(df["team_name"].astype(str))
X_track = track_le.transform(df["track_name"].astype(str))
X_start_comp = start_le.transform(df["starting_compound"].astype(str))

NUM_COLS = ["starting_position", "air_temp_mean", "track_temp_mean", "humidity_mean", "rain_minutes_ratio", "wind_speed_mean"]
X_num = df[NUM_COLS].values.astype("float32")

# Robust standardization (clip outliers)
for i in range(X_num.shape[1]):
    col = X_num[:, i]
    q1, q99 = np.percentile(col, [1, 99])
    col = np.clip(col, q1, q99)
    mean, std = col.mean(), col.std()
    if std > 0:
        X_num[:, i] = (col - mean) / std
    else:
        X_num[:, i] = 0.0

# Weather sequences padding
max_timesteps = max(s.shape[0] for s in X_seq_raw)
X_seq = pad_sequences(X_seq_raw, maxlen=max_timesteps, padding="post", dtype="float32")

# Normalize sequences
seq_mean = X_seq.mean(axis=(0, 1))
seq_std = X_seq.std(axis=(0, 1)) + 1e-8
X_seq = (X_seq - seq_mean) / seq_std

# Targets
y_stops = stops_le.transform(df["pit_stops"].astype(str))
y_tire = tire_le.transform(df["target_compound"].astype(str))
y_time = df["time_loss_target"].values.astype("float32")

# Log transform + normalize time
y_time_log = np.log1p(y_time)
y_time_mean = y_time_log.mean()
y_time_std = y_time_log.std() + 1e-8
y_time_norm = (y_time_log - y_time_mean) / y_time_std

# Split
indices = np.arange(len(df))
train_idx, val_idx = train_test_split(indices, test_size=0.20, random_state=42, stratify=y_stops)
print(f"Train: {len(train_idx)}, Val: {len(val_idx)}")

# Save preprocessing pack
preproc = {
    "team_le": team_le,
    "track_le": track_le,
    "start_le": start_le,
    "tire_le": tire_le,
    "stops_le": stops_le,
    "NUM_COLS": NUM_COLS,
    "seq_mean": seq_mean,
    "seq_std": seq_std,
    "y_time_mean": y_time_mean,
    "y_time_std": y_time_std,
    "max_timesteps": max_timesteps,
}
joblib.dump(preproc, PREPROC_FILE)
print("Saved preprocessing:", PREPROC_FILE)



Preprocessing...
Train: 696, Val: 174
Saved preprocessing: multitask_preprocessing_openf1_fastf1.joblib


## Build and train GRU multi-task model (same as improved)


In [9]:
print("\nBuilding model...")

EMBED_DIM = 16
GRU_UNITS = 64
DENSE_UNITS = 64
DROPOUT_RATE = 0.3
L2_REG = 0.001

weather_seq_in = layers.Input(shape=(max_timesteps, len(WEATHER_FEATURES)), name="weather_seq")
team_in = layers.Input(shape=(1,), dtype="int32", name="team_in")
track_in = layers.Input(shape=(1,), dtype="int32", name="track_in")
start_comp_in = layers.Input(shape=(1,), dtype="int32", name="start_comp_in")
num_in = layers.Input(shape=(len(NUM_COLS),), name="num_in")

team_emb = layers.Embedding(len(team_le.classes_), EMBED_DIM, embeddings_regularizer=regularizers.l2(L2_REG), name="team_emb")(team_in)
team_emb = layers.Flatten()(team_emb)

track_emb = layers.Embedding(len(track_le.classes_), EMBED_DIM, embeddings_regularizer=regularizers.l2(L2_REG), name="track_emb")(track_in)
track_emb = layers.Flatten()(track_emb)

start_emb = layers.Embedding(len(start_le.classes_), EMBED_DIM, embeddings_regularizer=regularizers.l2(L2_REG), name="start_comp_emb")(start_comp_in)
start_emb = layers.Flatten()(start_emb)

gru_out = layers.Bidirectional(
    layers.GRU(
        GRU_UNITS,
        return_sequences=False,
        dropout=DROPOUT_RATE,
        recurrent_dropout=0.2,
        kernel_regularizer=regularizers.l2(L2_REG),
    ),
    name="bi_gru",
)(weather_seq_in)

concat = layers.Concatenate(name="concat_all")([gru_out, team_emb, track_emb, start_emb, num_in])

shared = layers.Dense(DENSE_UNITS, activation="relu", kernel_regularizer=regularizers.l2(L2_REG), name="shared_dense_1")(concat)
shared = layers.Dropout(DROPOUT_RATE)(shared)
shared = layers.Dense(DENSE_UNITS // 2, activation="relu", kernel_regularizer=regularizers.l2(L2_REG), name="shared_dense_2")(shared)
shared = layers.Dropout(DROPOUT_RATE)(shared)

stops_out = layers.Dense(len(stops_le.classes_), activation="softmax", name="stops_out")(shared)
tire_out = layers.Dense(len(tire_le.classes_), activation="softmax", name="tire_out")(shared)
time_out = layers.Dense(1, activation="linear", name="time_out")(shared)

model = keras.Model(
    inputs={
        "weather_seq": weather_seq_in,
        "team_in": team_in,
        "track_in": track_in,
        "start_comp_in": start_comp_in,
        "num_in": num_in,
    },
    outputs={"stops_out": stops_out, "tire_out": tire_out, "time_out": time_out},
)

optimizer = keras.optimizers.Adam(learning_rate=0.0005, clipnorm=1.0)

model.compile(
    optimizer=optimizer,
    loss={
        "stops_out": "sparse_categorical_crossentropy",
        "tire_out": "sparse_categorical_crossentropy",
        "time_out": "mse",
    },
    loss_weights={
        "stops_out": 1.5,
        "tire_out": 1.0,
        "time_out": 0.8,
    },
    metrics={
        "stops_out": "accuracy",
        "tire_out": "accuracy",
        "time_out": "mae",
    },
)

model.summary()


Building model...


2026-05-12 01:50:50.916778: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2026-05-12 01:50:50.917156: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-05-12 01:50:50.917168: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2026-05-12 01:50:50.917213: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-05-12 01:50:50.917227: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ team_in             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ track_in            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ start_comp_in       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ weather_seq         │ (None, 290, 5)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ team_emb            │ (None, 1, 16)     │        304 │ team_in[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ track_emb           │ (None, 1, 16)     │        672 │ track_in[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ start_comp_emb      │ (None, 1, 16)     │         80 │ start_comp_in[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bi_gru              │ (None, 128)       │     27,264 │ weather_seq[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 16)        │          0 │ team_emb[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 16)        │          0 │ track_emb[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 16)        │          0 │ start_comp_emb[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ num_in (InputLayer) │ (None, 6)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_all          │ (None, 182)       │          0 │ bi_gru[0][0],     │
│ (Concatenate)       │                   │            │ flatten[0][0],    │
│                     │                   │            │ flatten_1[0][0],  │
│                     │                   │            │ flatten_2[0][0],  │
│                     │                   │            │ num_in[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dense_1      │ (None, 64)        │     11,712 │ concat_all[0][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ shared_dense_1[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dense_2      │ (None, 32)        │      2,080 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ shared_dense_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stops_out (Dense)   │ (None, 12)        │        396 │ dropout_1[0][0] 

 Total params: 42,706 (166.82 KB)

 Trainable params: 42,706 (166.82 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
print("\nTraining...")

X_train = {
    "weather_seq": X_seq[train_idx],
    "team_in": X_team[train_idx],
    "track_in": X_track[train_idx],
    "start_comp_in": X_start_comp[train_idx],
    "num_in": X_num[train_idx],
}

y_train = {
    "stops_out": y_stops[train_idx],
    "tire_out": y_tire[train_idx],
    "time_out": y_time_norm[train_idx],
}

X_val = {
    "weather_seq": X_seq[val_idx],
    "team_in": X_team[val_idx],
    "track_in": X_track[val_idx],
    "start_comp_in": X_start_comp[val_idx],
    "num_in": X_num[val_idx],
}

y_val = {
    "stops_out": y_stops[val_idx],
    "tire_out": y_tire[val_idx],
    "time_out": y_time_norm[val_idx],
}

callbacks = [
    EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=8, min_lr=1e-6, verbose=1),
    ModelCheckpoint(MODEL_FILE, monitor="val_loss", save_best_only=True, verbose=1),
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=150,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
)



Training...
Epoch 1/150


2026-05-12 01:50:53.495866: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


 3/22 ━━━━━━━━━━━━━━━━━━━━ 50:01 158s/step - loss: 7.1690 - stops_out_accuracy: 0.1389 - stops_out_loss: 2.5599 - time_out_loss: 1.7080 - time_out_mae: 0.9375 - tire_out_accuracy: 0.1389 - tire_out_loss: 1.8044

In [ ]:
# Plot training history (loss + task metrics)
hist = history.history

plt.figure(figsize=(14, 10))

plt.subplot(2, 2, 1)
plt.plot(hist.get("loss", []), label="train")
plt.plot(hist.get("val_loss", []), label="val")
plt.title("Total Loss")
plt.legend()

plt.subplot(2, 2, 2)
plt.plot(hist.get("stops_out_accuracy", []), label="train")
plt.plot(hist.get("val_stops_out_accuracy", []), label="val")
plt.title("Stops Accuracy")
plt.legend()

plt.subplot(2, 2, 3)
plt.plot(hist.get("tire_out_accuracy", []), label="train")
plt.plot(hist.get("val_tire_out_accuracy", []), label="val")
plt.title("Tire Accuracy")
plt.legend()

plt.subplot(2, 2, 4)
plt.plot(hist.get("time_out_mae", []), label="train")
plt.plot(hist.get("val_time_out_mae", []), label="val")
plt.title("Time MAE (normalized)")
plt.legend()

plt.tight_layout()
plt.show()

print("\n============================================================")
print("TRAINING COMPLETE")
print("============================================================")

eval_metrics = model.evaluate(X_val, y_val, verbose=0)
metric_names = model.metrics_names

print("\n============================================================")
print("FINAL VALIDATION METRICS")
print("============================================================")

for n, v in zip(metric_names, eval_metrics):
    if "accuracy" in n:
        print(f"{n}: {v*100:.2f}%")
    else:
        print(f"{n}: {v:.4f}")


## Strategy Simulator Output (core deliverable)

This section keeps the spirit of your original simulator:
- Load race context + forecast weather
- Enumerate 0..3-stop strategies and tire sequences
- Run model predictions and compute total race time ranking

Note: The *exact* race-time model is still a proxy unless you have lap-time baselines.
We keep it consistent with your original approach: predicted `time_loss` is key + pit loss + simple stint penalties.


In [ ]:
preproc = joblib.load(PREPROC_FILE)

def load_race_context(path=RACE_CONTEXT_FILE):
    ctx = safe_read_csv(Path(path))
    if ctx.empty:
        # Fallback: pick a representative row from training
        sample = df.sample(1, random_state=42).iloc[0].to_dict()
        ctx = pd.DataFrame([{
            "team_name": sample["team_name"],
            "track_name": sample["track_name"],
            "starting_position": sample["starting_position"],
            "starting_compound": sample["starting_compound"],
        }])
    return ctx

def load_forecast_weather(path=FORECAST_FILE):
    w = safe_read_csv(Path(path))
    if w.empty:
        # Fallback: reuse average weather from training
        w = pd.DataFrame([{c: float(df[f"{c.split('_')[0]}_temp_mean"].mean()) if "temperature" in c else float(df["wind_speed_mean"].mean()) for c in WEATHER_FEATURES}])

    # Accept either OpenF1 style columns or FastF1 style
    w = w.copy()
    # If forecast has OpenF1 names already: ok
    # If it has FastF1 names, remap
    rename_map = {
        "AirTemp": "air_temperature",
        "TrackTemp": "track_temperature",
        "Humidity": "humidity",
        "Rainfall": "rainfall",
        "WindSpeed": "wind_speed",
    }
    for src, dst in rename_map.items():
        if src in w.columns and dst not in w.columns:
            w[dst] = w[src]

    w = ensure_weather_columns(w)
    return w

race_ctx = load_race_context()
forecast_w = load_forecast_weather()

race_ctx.head(), forecast_w.head()

In [ ]:
def encode_inputs_for_model(team_name, track_name, starting_compound, starting_position, weather_seq):
    # Handle unseen labels by mapping to closest known or fallback
    def safe_transform(le, value, fallback_index=0):
        cls = list(le.classes_)
        if value in cls:
            return le.transform([value])[0]
        # fallback: first class
        return fallback_index

    team_idx = safe_transform(preproc["team_le"], str(team_name))
    track_idx = safe_transform(preproc["track_le"], str(track_name))
    start_idx = safe_transform(preproc["start_le"], normalize_compound(starting_compound))

    # numeric normalization must match training: we reuse means/stds indirectly by re-applying same robust step is hard.
    # Here we assume race_ctx numeric is already in the same scale as df pre-clip; we normalize by using training mean/std of each column from df.
    # Simple approach: recompute mean/std from df (post-clip) is not stored; we approximate with raw mean/std of df NUM_COLS.
    # If you want exact, store numeric scaler parameters explicitly.
    x_num = np.array([
        float(starting_position),
        float(df["air_temp_mean"].mean()),
        float(df["track_temp_mean"].mean()),
        float(df["humidity_mean"].mean()),
        float(df["rain_minutes_ratio"].mean()),
        float(df["wind_speed_mean"].mean()),
    ], dtype="float32")

    # Weather seq pad + normalize
    seq = np.asarray(weather_seq, dtype="float32")
    if seq.ndim != 2 or seq.shape[1] != len(WEATHER_FEATURES):
        raise ValueError(f"weather_seq must have shape [T, {len(WEATHER_FEATURES)}]")

    seq = pad_sequences([seq], maxlen=preproc["max_timesteps"], padding="post", dtype="float32")
    seq = (seq - preproc["seq_mean"]) / (preproc["seq_std"] + 1e-8)

    X = {
        "weather_seq": seq,
        "team_in": np.array([[team_idx]], dtype="int32"),
        "track_in": np.array([[track_idx]], dtype="int32"),
        "start_comp_in": np.array([[start_idx]], dtype="int32"),
        "num_in": np.array([x_num], dtype="float32"),
    }
    return X

def predict_race_outputs(X):
    preds = model.predict(X, verbose=0)
    p_stops = preds["stops_out"][0]
    p_tire = preds["tire_out"][0]
    t_norm = float(preds["time_out"][0][0])

    # denormalize time
    t_log = t_norm * preproc["y_time_std"] + preproc["y_time_mean"]
    t = float(np.expm1(t_log))

    return p_stops, p_tire, t

def enumerate_strategies(max_stops=3):
    # Enumerate sequences of compounds for stints given number of stops.
    # stints = stops + 1
    strategies = []
    for stops in range(0, max_stops + 1):
        stints = stops + 1
        # allow dry + wet; in practice you might condition on rainfall forecast
        for comps in product(VALID_COMPOUNDS, repeat=stints):
            strategies.append({"stops": stops, "compounds": comps})
    return strategies

def strategy_score(base_race_time, pit_loss_pred, pit_stops, compounds, rain_ratio):
    # Simple scoring:
    # total_time = base + pit_loss_pred + pit_stops * DEFAULT_PIT_LOSS + stint penalties
    # Stint penalty: discourage inappropriate compounds given rain forecast.
    total = float(base_race_time) + float(pit_loss_pred) + float(pit_stops) * DEFAULT_PIT_LOSS

    # Penalties
    penalty = 0.0
    for c in compounds:
        c = normalize_compound(c)
        if rain_ratio > 0.2:
            # rainy: penalize dry tyres
            if c in DRY_COMPOUNDS:
                penalty += 15.0
        else:
            # dry: penalize wet/inter
            if c in ["INTERMEDIATE", "WET"]:
                penalty += 10.0

    # Penalize repeating WET too much etc.
    total += penalty
    return total, penalty

def run_strategy_simulation(team_name, track_name, starting_position, starting_compound, forecast_weather_df, max_stops=3):
    # Build weather seq from forecast
    seq, ws = compute_weather_seq_and_summary(forecast_weather_df)
    if seq is None:
        raise ValueError("Forecast weather is empty / invalid")

    # Encode & predict
    X = encode_inputs_for_model(team_name, track_name, starting_compound, starting_position, seq)
    p_stops, p_tire, pit_loss_pred = predict_race_outputs(X)

    # Use most likely stop count as model suggestion
    stops_pred = int(np.argmax(p_stops))
    # map index back to class label
    try:
        stops_pred_label = int(preproc["stops_le"].inverse_transform([stops_pred])[0])
    except Exception:
        stops_pred_label = stops_pred

    # Use a baseline race time constant (relative ranking is what matters)
    base_race_time = 5400.0  # 90 minutes

    strategies = enumerate_strategies(max_stops=max_stops)
    out_rows = []

    rain_ratio = ws.get("rain_minutes_ratio", 0.0)

    for s in strategies:
        pit_stops = s["stops"]
        compounds = s["compounds"]

        total_time, penalty = strategy_score(base_race_time, pit_loss_pred, pit_stops, compounds, rain_ratio)

        out_rows.append({
            "pred_stops_argmax": stops_pred_label,
            "pit_loss_pred_sec": pit_loss_pred,
            "strategy_stops": pit_stops,
            "strategy_compounds": "-".join(compounds),
            "rain_ratio": rain_ratio,
            "total_time_sec": total_time,
            "penalty_sec": penalty,
        })

    out = pd.DataFrame(out_rows).sort_values("total_time_sec").reset_index(drop=True)

    # Add formatted time
    out["total_time"] = out["total_time_sec"].apply(format_time)
    return out, p_stops, p_tire

# Run for first row in race context
row = race_ctx.iloc[0]
team_name = row.get("team_name", df["team_name"].iloc[0])
track_name = row.get("track_name", df["track_name"].iloc[0])
starting_position = float(row.get("starting_position", 10.0))
starting_compound = normalize_compound(row.get("starting_compound", "MEDIUM"))

strategies_df, p_stops, p_tire = run_strategy_simulation(
    team_name=team_name,
    track_name=track_name,
    starting_position=starting_position,
    starting_compound=starting_compound,
    forecast_weather_df=forecast_w,
    max_stops=3,
)

print("\n============================================================")
print("STRATEGY OUTPUT (Ranked)")
print("============================================================")
print(f"Team: {team_name}")
print(f"Track: {track_name}")
print(f"Start Pos: {starting_position}")
print(f"Start Compound: {starting_compound}")

print("\nModel stop distribution (stops classes):")
stops_classes = list(preproc["stops_le"].classes_)
for i, cls in enumerate(stops_classes):
    print(f"  stops={cls}: {p_stops[i]:.3f}")

print("\nModel tire distribution (target compounds):")
tire_classes = list(preproc["tire_le"].classes_)
for i, cls in enumerate(tire_classes):
    print(f"  {cls}: {p_tire[i]:.3f}")

display(strategies_df.head(30))
